# n-step SARSA 算法

> n-step SARSA 是 **SARSA（TD(1)）与蒙特卡洛（MC）方法之间的统一桥梁**，通过引入超参数 $n$ 控制回望步数，在**偏差（Bias）与方差（Variance）之间取得灵活平衡**。
>
> **普通 SARSA（n=1）** 的更新目标仅向前看 1 步：
>
> $$G_t^{(1)} = r_{t+1} + \gamma Q(s_{t+1}, a_{t+1})$$
>
> **n-step SARSA** 将回望窗口扩展至 $n$ 步，TD 目标（n-step 回报）变为：
>
> $$G_t^{(n)} = r_{t+1} + \gamma r_{t+2} + \cdots + \gamma^{n-1} r_{t+n} + \gamma^n Q(s_{t+n}, a_{t+n})$$
>
> Q 值更新公式：
>
> $$Q(s_t, a_t) \leftarrow Q(s_t, a_t) - \alpha \left[Q(s_t, a_t) - G_t^{(n)}\right]$$
>
> 四种典型情形的对比如下：
>
> ---

| $n$ 取值 | 等价方法 | TD 目标 | 偏差 | 方差 |
|---|---|---|---|---|
| $n=1$，采样 $a'$ | 普通 SARSA（TD(1)） | $r_{t+1} + \gamma Q(s_{t+1}, a_{t+1})$ | 高（Bootstrap 引入） | 低 |
| $n=1$，期望 $a'$ | Expected SARSA | $r_{t+1} + \gamma \sum_{a'}\pi(a'\|s_{t+1})Q(s_{t+1},a')$ | 高（Bootstrap 引入） | **更低**（消除 $a'$ 采样噪声） |
| $1 < n < \infty$ | n-step SARSA | $\sum_{k=0}^{n-1}\gamma^k r_{t+k+1} + \gamma^n Q(s_{t+n}, a_{t+n})$ | 中 | 中 |
| $n \to \infty$ | 蒙特卡洛（MC） | $\sum_{k=0}^{T-t-1}\gamma^k r_{t+k+1}$（完整回报） | 低（无 Bootstrap） | 高 |

> ---
>
> **适当选择 $n$ 是 n-step 方法的核心**：$n$ 越大，Bootstrap 引入的偏差越小，但需等待更多步才能更新，方差也随之增大；$n=1$ 方差最低但偏差最大。实践中 $n \in [3, 10]$ 往往取得最佳权衡。

## 一、导入库与环境初始化

In [9]:
# 导入 NumPy 库，用于数组操作（Q 表、策略矩阵等）
import numpy as np
# 导入 random 模块，用于随机选择初始状态和动作
import random
# 导入 importlib 标准库，用于按文件路径动态加载模块
import importlib.util
# 导入 os 模块，用于拼接当前目录与文件名
import os
# 导入 time 模块（备用，可控制训练展示节奏）
import time
# 导入 Jupyter 显示控制函数，训练时清除输出以实时刷新展示
from IPython.display import clear_output

# 环境文件名 “02.1.ModelFree_Env_GridWorldV2.py” 以数字开头，
# Python 无法直接 import，需通过 importlib 按路径加载
_env_path = os.path.join(
    os.path.dirname(os.path.abspath("__file__")),
    "02.1.ModelFree_Env_GridWorldV2.py")  # 环境文件的绝对路径（str）
_spec = importlib.util.spec_from_file_location(
    "GridWorld_v2", _env_path)   # 构造模块规格对象
GridWorld_v2 = importlib.util.module_from_spec(_spec)  # 根据规格创建模块对象
_spec.loader.exec_module(GridWorld_v2)                  # 执行模块代码，完成加载

In [10]:
# 设置网格世界行数（必须与 desc 字符串行数一致）
rows = 5
# 设置网格世界列数（必须与 desc 每行字符数一致）
columns = 5

# 创建 5×5 GridWorld 环境实例
# forbiddenAreaScore=-10: 进入障碍格（'#'）的即时惩罚
# score=1: 到达目标格（'T'）的即时正奖励
# desc: 地图描述，'.'=普通格，'#'=障碍格，'T'=目标格
gridworld = GridWorld_v2.GridWorld_v2(
    forbiddenAreaScore=-10, score=1,
    desc=[".....", ".##..", "..#..", ".#T#.", ".#..."])

# 显示网格地图
gridworld.show()

# 初始化状态价值向量（全零，用于展示）
# shape: (25,)——每个元素对应一个状态（0~24）的价值估计
value = np.zeros(rows * columns)

# 初始化 Q 表（动作价值矩阵，全零）
# shape: (25, 5)——行=状态索引[0~24]，列=5种动作的 Q 值
qtable = np.zeros((rows * columns, 5))

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️


## 二、n-step SARSA 算法实现

In [11]:
def n_step_SARSA(gridworld: GridWorld_v2.GridWorld_v2,
                sarsaStep=5,
                epsilon=0.5,
                trajectorySteps=-1,
                learning_rate=0.001,
                final_epsilon=0.01,
                num_episodes=600,
                gamma=0.9) -> GridWorld_v2.GridWorld_v2:
    '''
    n-step SARSA 算法。

    在 SARSA 基础上，将 1 步 TD 误差扩展为 n 步累积回报，
    兼顾 TD 的低方差和 MC 的无偏性，可通过调整 n 来平衡偏差与方差。

    参数:
        gridworld (GridWorld_v2): GridWorld 环境实例
        sarsaStep (int): n-step 中的步数 n（n=1 退化为 SARSA，n→∞ 退化为 MC）
        epsilon (float): epsilon-greedy 初始探索率（0~1），越大越随机
        trajectorySteps (int): 每条轨迹最大步数；-1 表示到目标自动停止
        learning_rate (float): TD 学习率，控制 Q 值更新幅度
        final_epsilon (float): epsilon 的最小值，防止完全停止探索
        num_episodes (int): 训练轮数（每轮生成一条轨迹）
        gamma (float): 折扣因子，取值 (0,1]

    返回值:
        GridWorld_v2.GridWorld_v2: 训练后的环境对象（可查看最终策略）
    '''
    # 初始化状态价值向量（全零，仅用于展示）
    # shape: (25,)——每个元素对应一个状态的价值
    state_value = np.zeros((rows * columns))

    # 初始化动作价值矩阵 Q（全零）
    # shape: (25, 5)——行=状态[0~24]，列=5种动作的 Q 值
    action_value = np.zeros((rows * columns, 5))

    # 初始化随机确定性策略（独热编码）
    # shape: (25, 5)——每行只有一个位置为 1
    policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]

    for episode in range(num_episodes):  # 训练主循环
        print("episode", f"{episode}/{num_episodes}")  # 打印当前轮次

        # 线性衰减 epsilon（逐步从探索转向利用）
        if epsilon > final_epsilon:
            epsilon -= 0.001   # 每轮减少 0.001
        else:
            epsilon = final_epsilon  # 不低于最小探索率

        # 计算 epsilon-greedy 概率分配
        # p1：贪心动作的选取概率
        p1 = 1 - epsilon * (4/5)
        # p0：非贪心动作各自的选取概率
        p0 = epsilon / 5
        # 概率映射字典
        d = {1: p1, 0: p0}
        print("p1", p1, "p0", p0)  # 打印当前概率值

        # 将确定性策略转为 epsilon-greedy 概率分布
        # shape: (25, 5)——每行是一个状态的动作概率分布
        policy_epsilon = np.vectorize(d.get)(policy)

        # 初始化状态访问计数器（长度 25，记录每个状态的访问次数）
        cnt = [0 for i in range(25)]

        # 每轮随机选取初始状态（从所有可达自由格中均匀采样），
        # 确保状态空间被充分覆盖，避免固定起点导致远处格子 Q 值无法收敛
        # 可达自由格：排除障碍格（6,7,12,16,18,21）和目标格（17）
        reachable_states = [0,1,2,3,4,5,8,9,10,11,13,14,15,19,20,22,23,24]  # list[int]
        initState = random.choice(reachable_states)  # 随机初始状态（int）
        # 随机选择初始动作
        initAction = random.randint(0, 4)  # 随机初始动作（int，0~4）

        if trajectorySteps == -1:
            stop_when_reach_target = True  # 到达目标时自动停止

        # 按 epsilon-greedy 策略生成轨迹并计算每步奖励
        # 返回值：列表，每个元素为 (状态, 动作, 奖励, 下一状态, 下一动作) 五元组
        Trajectory = gridworld.getTrajectoryScore(
            nowState=initState,
            action=initAction,
            policy=policy_epsilon,
            steps=trajectorySteps,
            stop_when_reach_target=True)

        # 在轨迹末尾追加 sarsaStep 个目标格自循环，确保后 n 步的 Q 值更新有来源
        for i in range(sarsaStep):
            # (17, 4, 1, 17, 4)：目标格自循环（状态17，动作4，奖励1，仍在状态17）
            Trajectory.append((17, 4, 1, 17, 4))

        print("trajectorySteps", len(Trajectory))  # 打印轨迹总长度（含追加步）

        steps = len(Trajectory)  # 轨迹总长度 = 有效步数 + sarsaStep

        # 计算整条轨迹的折扣累积奖励（用于打印展示，不参与 Q 值更新）
        g = 1          # 折扣累积因子初始值
        reward = 0     # 累积奖励初始值
        for k in range(0, steps, 1):  # 遍历整条轨迹
            tmpstate, tmpaction, tmpscore, nextState, nextAction = Trajectory[k]
            reward += tmpscore * g  # 累加折扣奖励 r_k * γ^k
            g *= gamma              # 更新折扣因子
        print("Trajectory_score", reward)  # 打印轨迹总折扣奖励

        # n-step SARSA 核心更新循环
        # 对轨迹中每个有效时刻 t（从 0 到 steps-n），计算 n 步回报并更新 Q 值
        for k in range(0, steps - sarsaStep + 1, 1):
            cnt[Trajectory[k][0]] += 1  # 记录 t=k 时刻状态的访问次数

            # 初始化当前时刻的状态和动作（-1 表示尚未赋值）
            nowState, nowAction = -1, -1
            g = 1      # n 步折扣因子，初始为 1（即 γ^0）
            reward = 0  # n 步累积奖励初始为 0

            for i in range(sarsaStep):  # 向前展望 n 步
                # 解包轨迹中第 k+i 步的转移信息
                tmpstate, tmpaction, tmpscore, nextState, nextAction = Trajectory[k + i]

                # 仅在第一步（i=0）时记录起始状态和动作
                if nowState == -1:
                    nowState, nowAction = tmpstate, tmpaction  # 记录初始状态和动作

                # 累加折扣奖励：reward += r_{k+i} * γ^i
                reward += tmpscore * g
                # 更新折扣因子（下一步乘以 γ）
                g *= gamma

            # 加上第 n 步之后的自举值：G_t^(n) += γ^n * Q(s_{t+n}, a_{t+n})
            reward += g * action_value[nextState][nextAction]

            # n-step TD 误差：Q(s_t, a_t) - G_t^(n)
            TD_error = action_value[nowState][nowAction] - reward
            # 按 TD 误差更新 Q 值（朝减小误差方向修正）
            action_value[nowState][nowAction] -= learning_rate * TD_error

        # 策略改进：选 Q 值最大的动作作为新贪心策略
        # np.argmax(action_value, axis=1)：shape (25,)，每个状态的最优动作索引
        policy = np.eye(5)[np.argmax(action_value, axis=1)]  # shape (25, 5)
        # 更新 epsilon-greedy 策略
        policy_epsilon = np.vectorize(d.get)(policy)

        print(np.array(cnt).reshape(5, 5))  # 打印 5×5 状态访问次数矩阵

        # 计算状态价值 V(s) = Σ_a π(a|s)*Q(s,a)，shape (25,)
        state_value = np.sum(policy_epsilon * action_value, axis=1)
        # 全局平均状态价值
        mean_state_value = np.sum(policy_epsilon * action_value, axis=1).mean()

        gridworld.showPolicy(policy)                                   # 显示当前策略
        print(np.round(action_value, decimals=3).reshape(-1, 5))      # 打印 Q 表（精确到3位小数）
        print(np.round(state_value, decimals=4).reshape(5, 5))        # 打印 5×5 状态价值
        print("mean_state_value", mean_state_value)                    # 打印平均价值

    return gridworld  # 返回训练后的环境对象

## 三、最佳超参数训练

### 3.1 使用最佳超参数运行 n-step SARSA

> 根据超参数搜索（128 组合 × 3 随机种子，网格搜索结果），第 1 名参数如下：
>
> ---

| 参数 | 最佳值 | 说明 |
|---|---|---|
| `sarsaStep` | **3** | n-step 步数，兼顾偏差与方差的最优点 |
| `learning_rate` | **0.1** | 较大学习率，收敛速度快 |
| `gamma` | **0.9** | 折扣因子，0.9 可防止禁止格捷径污染 Q 值（详见下方代码注释） |
| `epsilon` | **0.8 → 0.01** | 先大胆探索，逐步衰减为贪心利用 |
| `num_episodes` | **1000** | 第 790 轮 epsilon 降至 0.01，后 210 轮接近纯贪心，确保收敛 |

> ---
>
> 搜索结果：成功率 **100%**，平均 **4.0 步**到达目标，综合评分 **0.9960**。

In [12]:
# 使用超参数搜索得到的最佳参数运行 n-step SARSA
# ⚠️ gamma 必须用 0.9（不能用 0.99）——本质原因如下：
#
#  「目标诱惑」现象：
#    n-step SARSA 在轨迹末尾追加目标格自循环（带正奖励），
#    相当于在终点处持续释放一个“永不衰减的正奖励信号”。
#    gamma 越接近 1，这个信号被折扣越少，能小步小步地反向传播得越远，
#    最终把目标格的 Q 值不断“撑高”。
#
#  为什么会穿墙：
#    禁止格紧邻在目标旁边，穿过禁止格是达到目标的“捷径”。
#    目标格 Q 值被撑得足够高时，即使穿越禁止格会受到惩罚，
#    捷径未来的投影奖励（来自被撑高的目标 Q）仍能补偿这些惩罚，
#    让穿墙捷径的总回报反而变成正值——策略于是主动选择穿墙。
#
#  低 gamma 为何正确：
#    gamma 小时，远处奖励被大幅折扣，目标的高 Q 值反向传播不了多远。
#    禁止格的惩罚是即时发生的、不折扣的；
#    而穿墙后取得的目标奖励已被折扣得很小。
#    “即时惩罚” 压过 “折扣后的未来奖励”，策略才能学会绕开禁止格行走。
n_step_SARSA(
    gridworld,
    sarsaStep=3,           # n-step 步数（int），偏差/方差平衡最优点
    learning_rate=0.1,     # 学习率 α（float），较大值加速收敛
    gamma=0.9,             # 折扣因子 γ（float），0.9 可防止禁止格捷径污染 Q 值
    epsilon=0.8,           # 初始探索率（float），鼓励充分探索
    final_epsilon=0.01,    # epsilon 衰减下限（float）
    num_episodes=1000,     # 训练轮数（int），epsilon 在第 790 轮降至 0.01，后 210 轮接近纯贪心
    trajectorySteps=5000,  # 每条轨迹最大步数（int），防止无限游荡
)

episode 0/1000
p1 0.3607999999999999 p0 0.1598
trajectorySteps 261
Trajectory_score -10.855869202763726
[[ 0  0  5 30 20]
 [ 0  0  4 32 26]
 [ 0  1  6 29 37]
 [ 0  0  2 13 31]
 [ 0  0  0  5 18]]
⬆️⬆️⬆️➡️⬆️
⬆️⏫️⏫️🔄⬅️
⬆️⬆️⏬⬆️⬆️
⬆️⏫️✅⏪⬆️
⬆️⏫️⬆️⬅️⬇️
[[ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.    -0.115 -1.906  0.    -0.078]
 [-1.236 -0.234 -0.986 -1.419 -0.81 ]
 [-0.107 -0.696 -0.184 -0.266 -0.146]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.    -0.093  0.     0.    -1.007]
 [-0.215 -0.304 -1.466 -1.013  0.   ]
 [-0.345 -0.663 -0.611 -0.034 -0.104]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.    -1.819  0.     0.     0.   ]
 [-1.184 -0.81  -0.719 -1.71  -3.481]
 [-0.228 -0.528 -2.809 -4.473 -3.187]
 [-0.125 -0.721 -2.276 -2.395 -2.213]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [-0.829  0.     0.     0.     0.271]
 [-0.931 -0.176 -1.787  0.    -1.   ]
 [-0.795 -1.673 -0.885 -5.953 -2